# timm
1,000여 개의 이미지 모델을 한 줄로

모델 이름을 와일드카드로 검색하고, pretrained=True로 사전학습 가중치까지 로드

In [ ]:
!pip install timm
import timm

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, "/ device:", device)

2.11.0+cu128 / device: cuda


In [3]:
# 사용 가능한 모델 검색 (와일드카드)
print(timm.list_models("*convnext*")[:5])
print(timm.list_models("*efficientnetv2*", pretrained=True)[:5])

# 모델 개수 확인
print(len(timm.list_models())) # 1000+
print(len(timm.list_models(pretrained=True)))

# 특정 모델의 기본 입력 크기 등 설정 확인
cfg = timm.get_pretrained_cfg("convnext_tiny")
print(cfg.input_size)

['convnext_atto', 'convnext_atto_ols', 'convnext_atto_rms', 'convnext_base', 'convnext_femto']
['efficientnetv2_rw_m.agc_in1k', 'efficientnetv2_rw_s.ra2_in1k', 'efficientnetv2_rw_t.ra2_in1k', 'gc_efficientnetv2_rw_t.agc_in1k', 'tf_efficientnetv2_b0.in1k']
1288
1703
(3, 224, 224)


## 사전 학습 모델 로드

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# EfficientNetV2-S를 개/고양이 2클래스용으로 로드
model = timm.create_model("tf_efficientnetv2_s", pretrained=True, num_classes=2) # 분류기 자동 교체
model = model.to(device)

# 특징 추출부만 동결 (선택)
for name, p in model.named_parameters():
  if "classifier" not in name:
    p.requires_grad = False

# 모델이 기대하는 전처리 확인 → transforms에 반영
data_cfg = timm.data.resolve_model_data_config(model)
print(data_cfg["input_size"], data_cfg["mean"], data_cfg["std"])

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

(3, 300, 300) (0.5, 0.5, 0.5) (0.5, 0.5, 0.5)


## 데이터셋 준비

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
transforms.Resize((256, 256)),
transforms.RandomCrop(224),
transforms.RandomHorizontalFlip(),
transforms.ToTensor(),
transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.ToTensor(),
transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

def binary_target(y):
  return 0 if y < 12 else 1


train_full = OxfordIIITPet(
    root="data", split="trainval", target_types="category",
    target_transform=binary_target,
    transform=train_transform, download=True
    )

test_set = OxfordIIITPet(
    root="data", split="test", target_types="category",
    target_transform=binary_target,
    transform=eval_transform, download=True
    )

n_val = int(len(train_full) * 0.2)
train_set, val_set = random_split(train_full,
[len(train_full) - n_val, n_val])

BATCH_SIZE = 32

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
shuffle=True, num_workers=0, pin_memory=True)

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE,
shuffle=False, num_workers=0, pin_memory=True)

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE,
shuffle=False, num_workers=0, pin_memory=True)

images, labels = next(iter(train_loader))



100%|██████████| 792M/792M [01:06<00:00, 11.9MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 13.2MB/s]


## 동일 파이프라인에서 모델만 교체하기

In [6]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm # for progress bar

# Define train_one_epoch function
def train_one_epoch(model, data_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(data_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(data_loader.dataset)

# Define evaluate function
def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(data_loader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

CANDIDATES = ["resnet50", "tf_efficientnetv2_s", "convnext_tiny"]
criterion = nn.CrossEntropyLoss()
results = {}

for name in CANDIDATES:
  model = timm.create_model(name, pretrained=True, num_classes=2).to(device)
  optimizer = optim.AdamW(model.parameters(), lr=1e-4)

  for epoch in range(1): # 시연용 1 에폭 fine-tuning
    train_one_epoch(model, train_loader, criterion, optimizer)

  acc = evaluate(model, val_loader)
  results[name] = acc
  print(f"{name:28s} val acc = {acc:.4f}")

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 23/23 [00:05<00:00,  4.18it/s]


resnet50                     val acc = 0.6970


Evaluating: 100%|██████████| 23/23 [00:05<00:00,  3.99it/s]


tf_efficientnetv2_s          val acc = 0.8628


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 23/23 [00:06<00:00,  3.62it/s]

convnext_tiny                val acc = 0.9293


#

## 결과 정리 — 정확도·크기·속도를 함께 기록

In [7]:
import time, pandas as pd

rows = []
for name, acc in results.items():
  model = timm.create_model(name, pretrained=True, num_classes=2).to(device).eval()
  n_params = sum(p.numel() for p in model.parameters()) / 1e6
  x = torch.randn(1, 3, 224, 224).to(device)
  t0 = time.time()

  with torch.no_grad():
    for _ in range(50): model(x)

  ms = (time.time() - t0) / 50 * 1000
  rows.append({"model": name, "val_acc": acc, "params(M)": round(n_params, 1), "infer(ms)": round(ms, 1)})

print(pd.DataFrame(rows).sort_values("val_acc", ascending=False))

                 model   val_acc  params(M)  infer(ms)
2        convnext_tiny  0.929348       27.8       45.1
1  tf_efficientnetv2_s  0.862772       20.2       18.4
0             resnet50  0.697011       23.5        7.5
